# Laboratorio 6: análisis de redes sociales en YouTube
## Incisos 1 y 2. Carga, integración, calidad y preprocesamiento

Este notebook resuelve los apartados 1.1 a 1.4 de las instrucciones. El objetivo es cargar los dos archivos, reconocer su estructura y sus llaves, explicar las relaciones entre las entidades e integrar cada comentario con los datos del video correspondiente.

Las consultas de búsqueda describen el procedimiento de recolección y no necesariamente el tema definitivo de cada video. Asimismo, `reply_count` no permite saber quién respondió a quién y, por tanto, no se interpreta aquí como una relación entre usuarios.

### 1.1 Carga de los archivos

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

# Permite ejecutar el notebook desde la carpeta del laboratorio o desde su subcarpeta.
directorios_candidatos = [Path.cwd(), Path.cwd().parent]
data_dir = next(
    (ruta for ruta in directorios_candidatos
     if (ruta / "youtube_videos.csv").exists()
     and (ruta / "youtube_comments.csv").exists()),
    None,
)
if data_dir is None:
    raise FileNotFoundError("No se encontraron ambos CSV en el directorio actual o su padre.")

ruta_videos = data_dir / "youtube_videos.csv"
ruta_comentarios = data_dir / "youtube_comments.csv"
print(f"Directorio de datos: {data_dir.resolve()}")

Directorio de datos: C:\Users\marti\OneDrive\Documentos\uni\s8\data science\labs\Lab 6


In [2]:
# Los identificadores se leen explícitamente como texto para no alterar su formato.
videos = pd.read_csv(
    ruta_videos,
    dtype={"video_id": "string", "channel_id": "string"},
)
comentarios = pd.read_csv(
    ruta_comentarios,
    dtype={
        "video_id": "string",
        "comment_id": "string",
        "channel_id": "string",
        "author_channel_id": "string",
    },
)

resumen_carga = pd.DataFrame(
    {
        "archivo": [ruta_videos.name, ruta_comentarios.name],
        "filas": [videos.shape[0], comentarios.shape[0]],
        "columnas": [videos.shape[1], comentarios.shape[1]],
    }
)
display(resumen_carga)

,archivo,filas,columnas
0,youtube_videos.csv,293,20
1,youtube_comments.csv,406,17


In [3]:
print("Muestra de videos")
display(videos[["video_id", "title", "channel_name", "category", "view_count"]].head(3))

print("Muestra de comentarios")
display(
    comentarios[["comment_id", "video_id", "author_name", "text", "reply_count"]].head(3)
)

Muestra de videos


,video_id,title,channel_name,category,view_count
0,-5puKGEqcUc,INSIVUMEH pronostica incremento de lluvias para el fin de semana en Guatemala,T13 Noticias Guatemala,News & Politics,2357
1,-E7OPOLjMug,BERNARDO ARÉVALO CALIFICA CAMBIO EN EL MP COMO EL FIN DE UNA ETAPA DE DETERI...,IDocumenta,People & Blogs,4
2,-KDglrIzRKo,¡HISTÓRICO! Mexico recupera petróleo robado por Guatemala... 🔔,México Poder,People & Blogs,29736


Muestra de comentarios


,comment_id,video_id,author_name,text,reply_count
0,Ugw-J65a1iYL9hqhELh4AaABAg,j43HgwYFKfk,@MarcosCarillo-b1r,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,0
1,Ugw-ZT9t9wU2V-tCaUZ4AaABAg,06mFNPU0aB8,@RaulPerez-cw2vi,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay polic...",0
2,Ugw0xaOb2CYXXoudtwJ4AaABAg,j43HgwYFKfk,@iamjimalesssa,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, e...",0


### 1.2 Unidad de observación, llave primaria y variables relevantes

**`youtube_videos.csv`**

- Unidad de observación: un video de YouTube.
- Llave primaria: `video_id`.
- Llave foránea conceptual: `channel_id`, que agrupa los videos publicados por un mismo canal.

**`youtube_comments.csv`**

- Unidad de observación: un comentario recolectado en un video.
- Llave primaria: `comment_id`.
- Llave foránea: `video_id`, que apunta al video comentado.
- Identificador del autor: `author_channel_id`. Este debe preferirse sobre `author_name` y `author_handle`, pues los nombres visibles pueden cambiar.

In [4]:
variables_relevantes = pd.DataFrame(
    [
        ("videos", "video_id", "Identificador único y llave primaria del video"),
        ("videos", "title", "Título y contenido descriptivo del video"),
        ("videos", "channel_id", "Identificador estable del canal propietario"),
        ("videos", "channel_name / channel_handle", "Etiquetas visibles del canal"),
        ("videos", "category", "Categoría asignada por YouTube"),
        ("videos", "source_query / query_hits / source_group", "Procedencia dentro del muestreo"),
        ("videos", "description / keywords", "Contenido textual y temático"),
        ("videos", "view_count", "Popularidad observada durante la recolección"),
        ("videos", "publish_date", "Fecha y hora de publicación"),
        ("comentarios", "comment_id", "Identificador único y llave primaria del comentario"),
        ("comentarios", "video_id", "Llave foránea del video comentado"),
        ("comentarios", "author_channel_id", "Identificador estable del autor"),
        ("comentarios", "author_name / author_handle", "Etiquetas visibles del autor"),
        ("comentarios", "text", "Contenido del comentario"),
        ("comentarios", "like_count_text / reply_count", "Interacciones observadas con el comentario"),
        ("comentarios", "source_query / source_group", "Procedencia dentro del muestreo"),
    ],
    columns=["archivo", "variable", "función en el análisis"],
)
display(variables_relevantes)

,archivo,variable,función en el análisis
0,videos,video_id,Identificador único y llave primaria del video
1,videos,title,Título y contenido descriptivo del video
2,videos,channel_id,Identificador estable del canal propietario
3,videos,channel_name / channel_handle,Etiquetas visibles del canal
4,videos,category,Categoría asignada por YouTube
5,videos,source_query / query_hits / source_group,Procedencia dentro del muestreo
6,videos,description / keywords,Contenido textual y temático
7,videos,view_count,Popularidad observada durante la recolección
8,videos,publish_date,Fecha y hora de publicación
9,comentarios,comment_id,Identificador único y llave primaria del comentario


La siguiente comprobación verifica empíricamente que las llaves propuestas no estén vacías ni repetidas. También revisa la completitud de las llaves foráneas necesarias para la integración.

In [5]:
validacion_llaves = pd.DataFrame(
    [
        ("videos", "video_id", videos["video_id"].isna().sum(), videos["video_id"].duplicated().sum()),
        ("comentarios", "comment_id", comentarios["comment_id"].isna().sum(), comentarios["comment_id"].duplicated().sum()),
        ("comentarios", "video_id (FK)", comentarios["video_id"].isna().sum(), comentarios["video_id"].duplicated().sum()),
        ("comentarios", "author_channel_id", comentarios["author_channel_id"].isna().sum(), comentarios["author_channel_id"].duplicated().sum()),
    ],
    columns=["archivo", "campo", "faltantes", "valores repetidos"],
)
display(validacion_llaves)

assert videos["video_id"].notna().all() and videos["video_id"].is_unique
assert comentarios["comment_id"].notna().all() and comentarios["comment_id"].is_unique
assert comentarios["video_id"].notna().all()

print("Las llaves primarias son completas y únicas; video_id está completo en comentarios.")

,archivo,campo,faltantes,valores repetidos
0,videos,video_id,0,0
1,comentarios,comment_id,0,0
2,comentarios,video_id (FK),0,387
3,comentarios,author_channel_id,0,74


Las llaves primarias son completas y únicas; video_id está completo en comentarios.


### 1.3 Relación entre los elementos

El modelo conceptual observado es:

```text
CANAL (channel_id) 1 ─── publica ─── N VIDEO (video_id)
AUTOR (author_channel_id) 1 ─── escribe ─── N COMENTARIO (comment_id)
VIDEO (video_id) 1 ─── recibe ─── N COMENTARIO (comment_id)
VIDEO N ─── pertenece a ─── 1 CATEGORÍA (category)
VIDEO/COMENTARIO N ─── fue recuperado mediante ─── CONSULTA (source_query)
```

- **Canal y video:** un canal puede publicar varios videos, mientras que cada video tiene un solo canal propietario en estos datos. `channel_id` identifica al canal; `channel_name` y `channel_handle` son etiquetas visibles.
- **Video y comentario:** cada comentario está asociado con un video mediante `video_id`, y un video puede tener ninguno o varios comentarios observados. Que un video no aparezca en comentarios no demuestra que no tenga comentarios en YouTube; puede deberse a la cobertura de la recolección.
- **Autor y comentario:** cada comentario tiene un autor identificado por `author_channel_id`, y un autor puede comentar una o varias veces. En YouTube las cuentas de autores también son canales, pero `author_channel_id` representa al autor del comentario y `channel_id` al propietario del video, por lo que sus roles no deben confundirse.
- **Categoría:** `category` es un atributo del video asignado por YouTube, no del autor ni del comentario. Todos los comentarios heredan la categoría de su video al efectuar la integración.
- **Consulta de búsqueda:** `source_query` documenta cómo se encontró el contenido. No es una clasificación temática definitiva. En videos, `query_hits` puede registrar varias consultas que recuperaron el mismo video; `source_group` distingue la estrategia general de búsqueda.
- **Respuestas:** `reply_count` solo es un conteo. No identifica autores ni relaciones directas entre usuarios.

### 1.4 Integración mediante `video_id`

Se usa una unión izquierda desde comentarios hacia videos porque la unidad final de observación debe seguir siendo el comentario. La validación `many_to_one` exige que muchos comentarios puedan apuntar a un video, pero impide que un `video_id` duplicado en la tabla de videos multiplique artificialmente las filas.

In [6]:
datos_integrados = comentarios.merge(
    videos,
    on="video_id",
    how="left",
    suffixes=("_comentario", "_video"),
    indicator=True,
    validate="many_to_one",
)

conteo_union = datos_integrados["_merge"].value_counts().reindex(
    ["both", "left_only", "right_only"], fill_value=0
)
asociados = int(conteo_union["both"])
no_asociados = int(conteo_union["left_only"])
porcentaje_asociado = asociados / len(comentarios) * 100 if len(comentarios) else 0

resumen_union = pd.DataFrame(
    {
        "métrica": [
            "Comentarios originales",
            "Comentarios asociados con un video",
            "Comentarios sin video asociado",
            "Porcentaje asociado",
            "Videos distintos con comentarios",
            "Filas después de la unión",
        ],
        "resultado": [
            len(comentarios),
            asociados,
            no_asociados,
            f"{porcentaje_asociado:.2f}%",
            comentarios["video_id"].nunique(),
            len(datos_integrados),
        ],
    }
)
display(resumen_union)

assert len(datos_integrados) == len(comentarios)
assert no_asociados == 0

,métrica,resultado
0,Comentarios originales,406
1,Comentarios asociados con un video,406
2,Comentarios sin video asociado,0
3,Porcentaje asociado,100.00%
4,Videos distintos con comentarios,19
5,Filas después de la unión,406


Los comentarios ya incluyen `video_title`, `channel_id` y `channel_name`. Como son campos redundantes, se comparan con los valores de la tabla de videos para detectar contradicciones. Para análisis posteriores conviene tomar de `youtube_videos.csv` los atributos del video y conservar los campos redundantes solo para auditoría.

In [7]:
consistencia = pd.DataFrame(
    {
        "comparación": [
            "video_title vs. title",
            "channel_id_comentario vs. channel_id_video",
            "channel_name_comentario vs. channel_name_video",
        ],
        "filas diferentes": [
            datos_integrados["video_title"].ne(datos_integrados["title"]).sum(),
            datos_integrados["channel_id_comentario"].ne(datos_integrados["channel_id_video"]).sum(),
            datos_integrados["channel_name_comentario"].ne(datos_integrados["channel_name_video"]).sum(),
        ],
    }
)
display(consistencia)

display(
    datos_integrados[[
        "comment_id", "video_id", "author_channel_id", "text",
        "title", "channel_id_video", "channel_name_video", "category",
        "view_count", "_merge",
    ]].head(5)
)

,comparación,filas diferentes
0,video_title vs. title,0
1,channel_id_comentario vs. channel_id_video,0
2,channel_name_comentario vs. channel_name_video,0


,comment_id,video_id,author_channel_id,text,title,channel_id_video,channel_name_video,category,view_count,_merge
0,Ugw-J65a1iYL9hqhELh4AaABAg,j43HgwYFKfk,UCdFlugHJJa4l3YqWuNRmvXw,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,La cooptación de Walter Mazariegos en la USAC,UCE4rsXcgDb6e1-a9iTbWzfg,Quorum,News & Politics,10156,both
1,Ugw-ZT9t9wU2V-tCaUZ4AaABAg,06mFNPU0aB8,UCvl1tzQeBeGy6efPTRJXSCw,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay polic...",Capturan a presuntos delincuentes disfrazados de mujer señalados de cometer ...,UCVpSRoZgngfSL03Nlbjtq9A,Noti7,News & Politics,6692,both
2,Ugw0xaOb2CYXXoudtwJ4AaABAg,j43HgwYFKfk,UCRAquv8el-tQ30bN7MlmySQ,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, e...",La cooptación de Walter Mazariegos en la USAC,UCE4rsXcgDb6e1-a9iTbWzfg,Quorum,News & Politics,10156,both
3,Ugw0xgUc2ISpBr5_T654AaABAg,j43HgwYFKfk,UCkbsS_3D-pvg1iHOld9uvIg,Veremos a este mafioso de Mazariegos en la cárcel y un buen tiempo en la som...,La cooptación de Walter Mazariegos en la USAC,UCE4rsXcgDb6e1-a9iTbWzfg,Quorum,News & Politics,10156,both
4,Ugw1ZzA21njWhaqQQTh4AaABAg,OkXlHx0hx-8,UCOLHH4ZpMxPYF-Q6e6wvn5A,eso es para que salga de USA por su propio pie \nque se auto deporten,EE.UU. envía a mexicanos deportados a Guatemala antes de su regreso a México...,UCRwA1NUcUnwsly35ikGhp0A,Noticias Telemundo,News & Politics,14200,both


### Conclusión del inciso 1

Los archivos contienen **293 videos con 20 variables** y **406 comentarios con 17 variables**. `video_id` y `comment_id` cumplen empíricamente las condiciones de unicidad y completitud de sus respectivas llaves primarias. La integración conserva el comentario como unidad de observación y respeta una relación muchos-a-uno con los videos.

Se asociaron **406 de 406 comentarios (100.00 %)** con un video; no quedó ningún comentario sin correspondencia. Los comentarios corresponden a **19 videos distintos** dentro del catálogo de 293 videos. Además, los campos redundantes de título, identificador de canal y nombre de canal no presentan diferencias entre ambos archivos. Estos resultados describen la cobertura del conjunto proporcionado y no implican que los demás videos carezcan de comentarios en YouTube.

# Inciso 2. Calidad, limpieza y preprocesamiento

La limpieza se realiza sobre copias para mantener intactos los DataFrames cargados en el inciso 1. Ninguna observación se elimina automáticamente: los faltantes, duplicados textuales y valores atípicos se documentan antes de decidir su uso en análisis posteriores.

## 2.1 Diagnóstico inicial de calidad

In [8]:
def diagnostico_columnas(df, archivo):
    return pd.DataFrame({
        "archivo": archivo,
        "variable": df.columns,
        "tipo observado": df.dtypes.astype(str).values,
        "faltantes": df.isna().sum().values,
        "porcentaje faltante": (df.isna().mean().mul(100).round(2)).values,
        "valores únicos": df.nunique(dropna=False).values,
    })

dimensiones = pd.DataFrame({
    "archivo": ["youtube_videos.csv", "youtube_comments.csv"],
    "filas": [len(videos), len(comentarios)],
    "columnas": [videos.shape[1], comentarios.shape[1]],
})
diagnostico = pd.concat([
    diagnostico_columnas(videos, "videos"),
    diagnostico_columnas(comentarios, "comentarios"),
], ignore_index=True)

display(dimensiones)
display(diagnostico)

,archivo,filas,columnas
0,youtube_videos.csv,293,20
1,youtube_comments.csv,406,17


,archivo,variable,tipo observado,faltantes,porcentaje faltante,valores únicos
0,videos,video_id,string,0,0.00,293
1,videos,title,object,0,0.00,274
2,videos,channel_name,object,0,0.00,97
3,videos,channel_id,string,0,0.00,97
4,videos,source_query,object,0,0.00,21
5,videos,source_group,object,0,0.00,3
6,videos,dataset_sources,object,0,0.00,23
7,videos,channel_handle,object,0,0.00,97
8,videos,published_time,object,13,4.44,81
9,videos,view_count_text,object,13,4.44,260


In [9]:
def variables_constantes(df):
    return [col for col in df.columns if df[col].nunique(dropna=False) == 1]

resumen_duplicados = pd.DataFrame({
    "archivo": ["videos", "comentarios"],
    "filas duplicadas exactas": [videos.duplicated().sum(), comentarios.duplicated().sum()],
    "llaves primarias duplicadas": [
        videos["video_id"].duplicated().sum(),
        comentarios["comment_id"].duplicated().sum(),
    ],
    "variables constantes": [variables_constantes(videos), variables_constantes(comentarios)],
})
display(resumen_duplicados)

print(f"Duplicados adicionales del texto original: {comentarios['text'].duplicated().sum()}")
print(f"Valores visualmente vacíos en like_count_text: {comentarios['like_count_text'].str.strip().eq('').sum()}")

,archivo,filas duplicadas exactas,llaves primarias duplicadas,variables constantes
0,videos,0,0,[]
1,comentarios,0,0,"[is_pinned, viewer_rating]"


Duplicados adicionales del texto original: 2
Valores visualmente vacíos en like_count_text: 189


In [10]:
def ids_con_multiples_etiquetas(df, id_col, etiqueta_col):
    return int(df.groupby(id_col, dropna=False)[etiqueta_col].nunique(dropna=False).gt(1).sum())

def etiquetas_con_multiples_ids(df, etiqueta_col, id_col):
    return int(df.groupby(etiqueta_col, dropna=False)[id_col].nunique(dropna=False).gt(1).sum())

consistencia_identidades = pd.DataFrame([
    ("channel_id con más de un channel_name", ids_con_multiples_etiquetas(videos, "channel_id", "channel_name")),
    ("channel_id con más de un channel_handle", ids_con_multiples_etiquetas(videos, "channel_id", "channel_handle")),
    ("channel_name asociado con más de un channel_id", etiquetas_con_multiples_ids(videos, "channel_name", "channel_id")),
    ("author_channel_id con más de un author_name", ids_con_multiples_etiquetas(comentarios, "author_channel_id", "author_name")),
    ("author_channel_id con más de un author_handle", ids_con_multiples_etiquetas(comentarios, "author_channel_id", "author_handle")),
    ("author_name asociado con más de un author_channel_id", etiquetas_con_multiples_ids(comentarios, "author_name", "author_channel_id")),
    ("channel_handle diferente de owner_handle", int(videos["channel_handle"].ne(videos["owner_handle"]).sum())),
    ("publish_date diferente de upload_date", int(videos["publish_date"].ne(videos["upload_date"]).sum())),
    ("título redundante diferente entre archivos", int(datos_integrados["video_title"].ne(datos_integrados["title"]).sum())),
    ("channel_id redundante diferente entre archivos", int(datos_integrados["channel_id_comentario"].ne(datos_integrados["channel_id_video"]).sum())),
    ("channel_name redundante diferente entre archivos", int(datos_integrados["channel_name_comentario"].ne(datos_integrados["channel_name_video"]).sum())),
], columns=["comprobación", "inconsistencias"])
display(consistencia_identidades)

,comprobación,inconsistencias
0,channel_id con más de un channel_name,0
1,channel_id con más de un channel_handle,0
2,channel_name asociado con más de un channel_id,0
3,author_channel_id con más de un author_name,0
4,author_channel_id con más de un author_handle,0
5,author_name asociado con más de un author_channel_id,0
6,channel_handle diferente de owner_handle,0
7,publish_date diferente de upload_date,0
8,título redundante diferente entre archivos,0
9,channel_id redundante diferente entre archivos,0


## 2.2 Variables problemáticas o de uso delicado

| Variable | Decisión y justificación |
|---|---|
| `viewer_rating` | Excluir de análisis: sus 406 valores están ausentes. |
| `is_pinned` | Conservar para auditoría, pero excluir como predictor o criterio comparativo porque es constante (`False`). |
| `upload_date` | Es redundante con `publish_date` en el 100 % de los videos. Se conserva, pero se usa `publish_date` como fecha principal. |
| `owner_handle` | Coincide con `channel_handle`; se conserva únicamente como campo redundante de auditoría. |
| `video_title`, `channel_name` y `channel_id` en comentarios | Son redundantes. Tras comprobar su consistencia, se prefieren los atributos provenientes de videos después de unir por `video_id`. |
| `published_time` y `published_text` | Son tiempos relativos al momento de extracción y no fechas exactas. No se convierten a fecha. |
| `view_count_text` | Es una representación visual, tiene faltantes y puede diferir de `view_count` por el momento de captura. Se convierte para auditoría, pero se usa `view_count` en cálculos. |
| `like_count_text` | Los espacios en blanco representan ausencia de un conteo visible y se codifican como cero; otros formatos inválidos quedan como faltantes. |
| `reply_count` | Solo indica cuántas respuestas recibió el comentario; no permite crear aristas entre autores. |
| `channel_name`, `author_name` y handles | Sirven como etiquetas, no como identificadores estables. Se mantienen los ID correspondientes. |
| `source_query` | Describe el muestreo, no necesariamente el tema del contenido. |
| `query_hits`, `keywords`, `dataset_sources` | Son listas serializadas o valores separados por `|`; deben convertirse antes de analizarlos. |
| `description_snippet` | Puede estar incompleta; para contenido se prefiere `description`. |

Los faltantes en descripciones no justifican eliminar videos completos. La cobertura de comentarios también exige cautela: los 406 comentarios corresponden a 19 de 293 videos, por lo que ausencia en este archivo no equivale a ausencia de comentarios en YouTube.

## 2.3 Normalización de identificadores y nombres

Los ID de YouTube pueden distinguir mayúsculas y minúsculas; por eso solo se eliminan espacios externos y nunca se cambian de caja ni se reemplazan por nombres. En nombres y handles se aplica Unicode NFKC, recorte y colapso de espacios internos.

In [11]:
import re
import unicodedata

videos_limpios = videos.copy()
comentarios_limpios = comentarios.copy()

def normalizar_id(serie):
    return serie.astype("string").str.strip()

def normalizar_etiqueta(valor):
    if pd.isna(valor):
        return pd.NA
    valor = unicodedata.normalize("NFKC", str(valor))
    return re.sub(r"\s+", " ", valor).strip()

ids_video = ["video_id", "channel_id"]
ids_comentario = ["video_id", "comment_id", "channel_id", "author_channel_id"]
etiquetas_video = ["channel_name", "channel_handle", "owner_handle"]
etiquetas_comentario = ["channel_name", "author_name", "author_handle"]

cambios_normalizacion = []
for col in ids_video:
    nueva = normalizar_id(videos_limpios[col])
    cambios_normalizacion.append(("videos", col, int(videos_limpios[col].astype("string").ne(nueva).sum())))
    videos_limpios[col] = nueva
for col in ids_comentario:
    nueva = normalizar_id(comentarios_limpios[col])
    cambios_normalizacion.append(("comentarios", col, int(comentarios_limpios[col].astype("string").ne(nueva).sum())))
    comentarios_limpios[col] = nueva
for col in etiquetas_video:
    nueva = videos_limpios[col].map(normalizar_etiqueta).astype("string")
    cambios_normalizacion.append(("videos", col, int(videos_limpios[col].astype("string").ne(nueva).sum())))
    videos_limpios[col] = nueva
for col in etiquetas_comentario:
    nueva = comentarios_limpios[col].map(normalizar_etiqueta).astype("string")
    cambios_normalizacion.append(("comentarios", col, int(comentarios_limpios[col].astype("string").ne(nueva).sum())))
    comentarios_limpios[col] = nueva

display(pd.DataFrame(cambios_normalizacion, columns=["archivo", "variable", "valores modificados"]))
assert videos_limpios["video_id"].is_unique
assert comentarios_limpios["comment_id"].is_unique
assert comentarios_limpios["video_id"].isin(videos_limpios["video_id"]).all()

,archivo,variable,valores modificados
0,videos,video_id,0
1,videos,channel_id,0
2,comentarios,video_id,0
3,comentarios,comment_id,0
4,comentarios,channel_id,0
5,comentarios,author_channel_id,0
6,videos,channel_name,2
7,videos,channel_handle,0
8,videos,owner_handle,0
9,comentarios,channel_name,0


## 2.4 Conversión de conteos almacenados como texto

La función siguiente elimina etiquetas como `vistas`, interpreta comas o puntos como separadores de miles cuando no hay abreviatura y admite sufijos `K`, `M` y `B`. Con abreviatura, una coma o punto se interpreta como decimal. Valores negativos o formatos desconocidos se convierten en `NA`. En `like_count_text`, un texto vacío se interpreta como cero porque YouTube no muestra el contador cuando no hay 'me gusta'.

In [12]:
def convertir_conteo(valor, vacio_es_cero=False):
    if pd.isna(valor):
        return 0 if vacio_es_cero else pd.NA

    texto = unicodedata.normalize("NFKC", str(valor)).lower().strip()
    if not texto:
        return 0 if vacio_es_cero else pd.NA

    texto = re.sub(r"\b(vistas?|views?|likes?|me gusta)\b", "", texto)
    texto = re.sub(r"\s+", "", texto)
    coincidencia = re.fullmatch(r"(\d+(?:[.,]\d+)*)([kmb])?", texto)
    if not coincidencia:
        return pd.NA

    numero_texto, sufijo = coincidencia.groups()
    if sufijo:
        # Con abreviatura, el último separador expresa decimales.
        partes = re.split(r"[.,]", numero_texto)
        numero = float("".join(partes[:-1]) + "." + partes[-1]) if len(partes) > 1 else float(partes[0])
        multiplicador = {"k": 1_000, "m": 1_000_000, "b": 1_000_000_000}[sufijo]
        return round(numero * multiplicador)

    # Sin abreviatura, los separadores son de miles.
    if not re.fullmatch(r"(?:\d+|\d{1,3}(?:[.,]\d{3})+)", numero_texto):
        return pd.NA
    return int(re.sub(r"[.,]", "", numero_texto))

videos_limpios["view_count_text_num"] = pd.Series(
    (convertir_conteo(v) for v in videos_limpios["view_count_text"]),
    index=videos_limpios.index, dtype="Int64",
)
comentarios_limpios["like_count"] = pd.Series(
    (convertir_conteo(v, vacio_es_cero=True) for v in comentarios_limpios["like_count_text"]),
    index=comentarios_limpios.index, dtype="Int64",
)

vistas_presentes = videos_limpios["view_count_text"].notna()
likes_no_vacios = comentarios_limpios["like_count_text"].str.strip().ne("")
auditoria_conteos = pd.DataFrame([
    ("view_count_text", int((~vistas_presentes).sum()), int((vistas_presentes & videos_limpios["view_count_text_num"].isna()).sum())),
    ("like_count_text", int((~likes_no_vacios).sum()), int((likes_no_vacios & comentarios_limpios["like_count"].isna()).sum())),
], columns=["variable", "vacíos o faltantes", "formatos no válidos"])
display(auditoria_conteos)

diferencias_vistas = videos_limpios.loc[vistas_presentes, "view_count_text_num"].ne(
    videos_limpios.loc[vistas_presentes, "view_count"]
).sum()
print(f"Conteos visuales de vistas distintos de view_count: {diferencias_vistas}")
print("Se conserva view_count como variable cuantitativa principal, según el diccionario de datos.")

,variable,vacíos o faltantes,formatos no válidos
0,view_count_text,13,0
1,like_count_text,189,0


Conteos visuales de vistas distintos de view_count: 53
Se conserva view_count como variable cuantitativa principal, según el diccionario de datos.


In [13]:
import ast

def convertir_lista_literal(valor):
    if pd.isna(valor) or not str(valor).strip():
        return []
    try:
        resultado = ast.literal_eval(str(valor))
        return resultado if isinstance(resultado, list) else [resultado]
    except (ValueError, SyntaxError):
        return []

videos_limpios["query_hits_lista"] = videos_limpios["query_hits"].map(convertir_lista_literal)
videos_limpios["keywords_lista"] = videos_limpios["keywords"].map(convertir_lista_literal)
videos_limpios["dataset_sources_lista"] = videos_limpios["dataset_sources"].str.split(r"\s*\|\s*", regex=True)
comentarios_limpios["dataset_sources_lista"] = comentarios_limpios["dataset_sources"].str.split(r"\s*\|\s*", regex=True)
videos_limpios["publish_date"] = pd.to_datetime(videos_limpios["publish_date"], utc=True, errors="coerce")
videos_limpios["upload_date"] = pd.to_datetime(videos_limpios["upload_date"], utc=True, errors="coerce")

print(f"Fechas publish_date no convertibles: {videos_limpios['publish_date'].isna().sum()}")
print(f"Fechas upload_date no convertibles: {videos_limpios['upload_date'].isna().sum()}")
display(videos_limpios[["query_hits_lista", "keywords_lista", "dataset_sources_lista"]].head(3))

Fechas publish_date no convertibles: 0
Fechas upload_date no convertibles: 0


,query_hits_lista,keywords_lista,dataset_sources_lista
0,[guatemala lluvias],"[Canícula prolongada, Chapin tv, Fenómeno del Niño, Guatemala, Lluvias en Gu...","[youtube_guatemala.csv, youtube_guatemala_lab.csv, youtube_guatemala_plus.csv]"
1,[@GobiernodelaRepublicadeGuatema],[],[youtube_target_channels.csv]
2,[guatemala noticias],"[Mexico recupera petroleo, petroleo robado Guatemala, huachicol frontera sur...","[youtube_guatemala.csv, youtube_guatemala_lab.csv, youtube_guatemala_plus.csv]"


### Valores atípicos

Se utiliza la regla descriptiva de 1.5 veces el rango intercuartílico (IQR). Los valores señalados no se eliminan: en redes sociales, distribuciones muy asimétricas y unos pocos contenidos populares son fenómenos plausibles. Cuando el IQR es cero, como ocurre con respuestas, todo valor positivo queda señalado por la regla; esto demuestra por qué una bandera estadística no equivale a un error de datos.

In [14]:
def resumen_atipicos(serie, variable):
    serie = pd.to_numeric(serie, errors="coerce").dropna()
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    bandera = (serie < inferior) | (serie > superior)
    return {
        "variable": variable, "mínimo": serie.min(), "Q1": q1,
        "mediana": serie.median(), "Q3": q3, "máximo": serie.max(),
        "límite superior IQR": superior, "atípicos IQR": int(bandera.sum()),
        "valores negativos": int((serie < 0).sum()),
    }

resumen_numerico = pd.DataFrame([
    resumen_atipicos(videos_limpios["view_count"], "view_count"),
    resumen_atipicos(comentarios_limpios["like_count"], "like_count"),
    resumen_atipicos(comentarios_limpios["reply_count"], "reply_count"),
])
display(resumen_numerico)

print("Videos con más visualizaciones (se conservan):")
display(videos_limpios.nlargest(5, "view_count")[["video_id", "title", "view_count"]])

,variable,mínimo,Q1,mediana,Q3,máximo,límite superior IQR,atípicos IQR,valores negativos
0,view_count,2,215.0,1175.0,7465.0,8190449,18340.0,49,0
1,like_count,0,0.0,1.0,2.0,405,5.0,48,0
2,reply_count,0,0.0,0.0,0.0,7,0.0,30,0


Videos con más visualizaciones (se conservan):


,video_id,title,view_count
148,THY2YBne3QE,"Los ALUCINANTES autobuses de Guatemala | ""Burras""",8190449
245,n4xHxJPNr78,🚫LOS FAMOSOS BUSES ESMERALDA los mas RÁPIDOS DE GUATEMALA 😱,3152619
113,KkNQhX_kg8A,🇬🇹HISTORIA de GUATEMALA en 17 minutos🇬🇹 - El Mapa de Sebas,749356
56,Aa7hIDCRh3g,Este BUS me LLEVO a UN PARAISO EN GUATEMALA,504374
126,NnKpTEfhWjI,✅ ASÍ es una EXHIBICION DE BUSES EN GUATEMALA,424874


## 2.5 y 2.6 Texto original y texto limpio

`texto_original` conserva exactamente `text` para auditoría y futuro análisis de sentimiento. `texto_limpio` está destinado a frecuencias, bigramas y tópicos, con estas decisiones:

| Operación | Tratamiento | Justificación |
|---|---|---|
| Minúsculas | Sí | Evita contar por separado variantes por caja. |
| URL | Se elimina la URL completa | Los enlaces suelen aportar ruido al vocabulario. |
| Hashtags | Se quita `#`, pero se conserva la palabra | La etiqueta puede aportar información temática. |
| Menciones | Se elimina el handle completo | Identifica cuentas, no contenido lingüístico; el original permite recuperarlo. |
| Puntuación | Se elimina | Facilita la tokenización para frecuencias. |
| Números | Se eliminan | Sin normalización contextual, suelen fragmentar el vocabulario. |
| Stopwords | Se usa una lista explícita en español | Reduce palabras funcionales frecuentes y mantiene reproducibilidad sin descargas. |
| Lematización | No se aplica | No hay un modelo morfológico español validado entre las dependencias del proyecto; reglas ad hoc podrían alterar nombres y significado. Puede añadirse después con un modelo versionado. |
| Emojis | Se eliminan de `texto_limpio` | Para tópicos actúan como tokens no léxicos. Permanecen en `texto_original`, indispensable para sentimiento. |

No se eliminan acentos ni la `ñ`, porque son rasgos ortográficos significativos en español.

In [15]:
import html

STOPWORDS_ES = {
    "a", "al", "algo", "algunas", "algunos", "ante", "antes", "aquel", "aquella",
    "aquellas", "aquellos", "aquí", "así", "aunque", "bajo", "bien", "cada", "casi",
    "como", "con", "contra", "cual", "cuando", "de", "del", "desde", "donde", "dos",
    "durante", "e", "el", "ella", "ellas", "ellos", "en", "entre", "era", "eran",
    "eres", "es", "esa", "esas", "ese", "eso", "esos", "esta", "estaba", "están",
    "estar", "este", "esto", "estos", "fue", "han", "hasta", "hay", "la", "las",
    "le", "les", "lo", "los", "más", "me", "mi", "mis", "mucho", "muy", "nada",
    "ni", "no", "nos", "nuestra", "nuestro", "o", "otra", "otro", "para", "pero",
    "poco", "por", "porque", "que", "qué", "quien", "se", "sea", "ser", "si", "sí",
    "sin", "sobre", "son", "su", "sus", "también", "te", "tiene", "todo", "tu", "tus",
    "un", "una", "uno", "unos", "usted", "ustedes", "ya", "y", "yo",
}

def limpiar_comentario(valor):
    if pd.isna(valor):
        return ""
    texto = unicodedata.normalize("NFKC", html.unescape(str(valor))).lower()
    texto = re.sub(r"(?:https?://|www\.)\S+", " ", texto)
    texto = re.sub(r"(?<!\w)@[\w.-]+", " ", texto)
    texto = re.sub(r"#(?=\w)", "", texto)
    texto = re.sub(r"\d+", " ", texto)
    texto = re.sub(r"[^a-záéíóúüñ\s]", " ", texto)
    tokens = [token for token in texto.split() if token not in STOPWORDS_ES]
    return " ".join(tokens)

comentarios_limpios["texto_original"] = comentarios_limpios["text"].astype("string")
comentarios_limpios["texto_limpio"] = comentarios_limpios["texto_original"].map(limpiar_comentario).astype("string")

display(comentarios_limpios[["comment_id", "texto_original", "texto_limpio"]].head(8))

,comment_id,texto_original,texto_limpio
0,Ugw-J65a1iYL9hqhELh4AaABAg,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,corrupto amigo vieja fiscal tengo verbose carcel
1,Ugw-ZT9t9wU2V-tCaUZ4AaABAg,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay polic...",jóvenes buscan trabajo tuvieron suerte policías gusta vando ir vestido mujer...
2,Ugw0xaOb2CYXXoudtwJ4AaABAg,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, e...",dejaron ganas demandar ilegalidad reuniones virtuales maquila
3,Ugw0xgUc2ISpBr5_T654AaABAg,Veremos a este mafioso de Mazariegos en la cárcel y un buen tiempo en la som...,veremos mafioso mazariegos cárcel buen tiempo sombra
4,Ugw1ZzA21njWhaqQQTh4AaABAg,eso es para que salga de USA por su propio pie \nque se auto deporten,salga usa propio pie auto deporten
5,Ugw1ZzA21njWhaqQQTh4AaABAg.Aa9Tf29tBouAaB_XZQglTL,Imagine if they had to walk back home.,imagine if they had to walk back home
6,Ugw2014OIYf336oytFt4AaABAg,buenísima investigacion :hand-purple-blue-peace::hand-purple-blue-peace::han...,buenísima investigacion hand purple blue peace hand purple blue peace hand p...
7,Ugw2bSVOoNn2pc8ZcUl4AaABAg,"Lleven su lonchera, sacrifiquense un poco. Y reintevren ese dinero. O están ...",lleven lonchera sacrifiquense reintevren dinero robando


## 2.7 Efecto de la limpieza

Se reportan duplicados como filas adicionales después de la primera aparición. Los duplicados no se eliminan porque comentarios idénticos con `comment_id` distinto siguen siendo actos de participación distintos y serán necesarios para ponderar aristas.

In [16]:
original = comentarios_limpios["texto_original"].fillna("")
limpio = comentarios_limpios["texto_limpio"].fillna("")
efecto_limpieza = pd.DataFrame({
    "métrica": [
        "Registros iniciales", "Registros finales", "Registros eliminados",
        "Textos modificados", "Textos vacíos antes", "Textos vacíos después",
        "Duplicados adicionales antes", "Duplicados adicionales después",
    ],
    "resultado": [
        len(original), len(limpio), len(original) - len(limpio),
        int(original.ne(limpio).sum()), int(original.str.strip().eq("").sum()),
        int(limpio.str.strip().eq("").sum()), int(original.duplicated().sum()),
        int(limpio.duplicated().sum()),
    ],
})
display(efecto_limpieza)

assert len(comentarios_limpios) == len(comentarios)
assert comentarios_limpios["texto_original"].equals(comentarios["text"].astype("string"))

,métrica,resultado
0,Registros iniciales,406
1,Registros finales,406
2,Registros eliminados,0
3,Textos modificados,403
4,Textos vacíos antes,0
5,Textos vacíos después,6
6,Duplicados adicionales antes,2
7,Duplicados adicionales después,11


In [17]:
datos_integrados_limpios = comentarios_limpios.merge(
    videos_limpios,
    on="video_id",
    how="left",
    suffixes=("_comentario", "_video"),
    validate="many_to_one",
)
assert len(datos_integrados_limpios) == len(comentarios_limpios)
assert datos_integrados_limpios["title"].notna().all()
print(f"Dataset integrado y preprocesado: {datos_integrados_limpios.shape[0]} filas y {datos_integrados_limpios.shape[1]} columnas.")

Dataset integrado y preprocesado: 406 filas y 44 columnas.


## Conclusión del inciso 2

La revisión conserva las 293 observaciones de videos y los 406 comentarios. No se encontraron filas completas ni llaves primarias duplicadas, y los cruces de ID, nombres y handles resultaron consistentes. `viewer_rating` no puede utilizarse y `is_pinned` no discrimina observaciones. Los faltantes de texto descriptivo se mantienen explícitos y los atípicos de participación no se borran, pues pueden representar concentración real.

Los identificadores estables se preservan, los conteos textuales quedan disponibles en formato entero y las listas/fechas tienen representaciones aptas para análisis. De los conteos visuales, 13 valores de vistas están ausentes, 53 difieren de `view_count` y los 189 campos vacíos de 'me gusta' se interpretaron como cero; no hubo formatos inválidos. Por ello, `view_count` continúa siendo la fuente principal para visualizaciones.

Cada comentario conserva su contenido exacto en `texto_original` y añade `texto_limpio` para análisis léxico. La transformación modificó 403 textos; 6 quedaron sin tokens útiles y los duplicados adicionales pasaron de 2 a 11. No se eliminó ninguna observación. El resultado integrado se almacena en `datos_integrados_limpios`.